# 05 · Oracle labels and the evaluator comparison (Table 6)

**Runtime: CPU is fine.** Everything reads the caches on Drive (generations from notebook 01, embeddings from its free-text cell).

| Part | What it does | Output on Drive (`MyDrive/mpac_hi_pt/`) |
|---|---|---|
| A · Oracle sheets | Samples generations and writes one sheet per annotator, per language | `oracle/<round>/<lang>/<annotator>.csv` |
| B · Agreement | Reads the filled sheets, Cohen's kappa per language | (printed) |
| C · Table 6 | Thresholds tuned on **tune** labels, then systems 4, 5, 7 on the same outputs and budget, scored on **test** labels | `results/<model>/<checkpoint>/table6_<source>.json` |

Protocol: `docs/oracle_protocol.md`. Order: **pilot** round (30 rows, agreement only, never used as labels), then **main** round. Part C can run before any human labels with `SOURCE = "proxy"` (TruthfulQA answers, honesty only) to check the pipeline; those numbers are **not reportable**.

System 6 (LLM-as-judge) isn't in the table yet (D5).

In [ ]:
# Clone (or update) the repo and install it. Re-run after a runtime reset.
import os, subprocess
REPO_DIR = "/content/mpac"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hurshprasad/Multi-Agent-Diagnosis-of-Multilingual-Pluralistic-Alignment.git $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR
!pip install -q -e ".[signals]"
import sys
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs aren't always visible to a running kernel

In [ ]:
QUICK = False
MODEL = "qwen2.5-7b"
CONFIGS = ["configs/hi_pt.yaml"] + (["configs/hi_pt_quick.yaml"] if QUICK else [])

from google.colab import drive
drive.mount("/content/drive")
from mpac import load_config
from mpac.config import get_model
from mpac.paths import resolve_root, subdir
cfg = load_config(*CONFIGS)
m = get_model(cfg, None if QUICK else MODEL)
print("model:", m["id"], "| languages:", cfg["languages"], "| artifacts ->", resolve_root(cfg))

## A · Oracle sheets

Run once per round: first `ROUND = "pilot"` (30 rows), then, once the pilot's kappa is ≥ 0.6, `ROUND = "main"` (`data.n_oracle_labels`, 120 for hi/pt). The main round never re-draws pilot rows, and an existing round is never overwritten.

- `ANNOTATORS`: two per language, fluent in English and that language. Use short names without spaces; each gets their own file (and their own row order).
- `LABEL_CHECKPOINTS`: **base only** for the hi/pt pilot. 120 labels spread over three checkpoints would leave about 4 test labels per cell per checkpoint, too few to score anything.

**Send each annotator** their CSV (Drive → right-click → Share, or download it) and `docs/oracle_protocol.md`. They fill `aligned` and `collapsed_vs_reference` with yes / no (and `notes` if needed), then **download as CSV** (File → Download → .csv in Google Sheets) and you put it back as `oracle/<round>/<lang>/filled/<annotator>.csv`, same file name. The adjudicator's sheet goes in the same folder as `adjudication.csv`.

In [ ]:
from mpac.config import all_languages
from mpac.data.loaders import load_items
from mpac.data.oracle import export_round, round_keys, sample_for_labelling
from mpac.data.splits import get_or_create_splits
from mpac.generation.cache import GenerationCache

ROUND = "pilot"   # "pilot" (30 rows) first, then "main"
ANNOTATORS = {"hi": ["hi_annotator1", "hi_annotator2"], "pt": ["pt_annotator1", "pt_annotator2"]}
LABEL_CHECKPOINTS = ["base"]

items = load_items(cfg)
splits = get_or_create_splits(cfg, items)
cache = GenerationCache(subdir(cfg, "generations"))
gens = {(m["id"], ck): cache.load_all(m["id"], ck, all_languages(cfg)) for ck in LABEL_CHECKPOINTS}
oracle_dir = subdir(cfg, "oracle")
earlier = set().union(*(round_keys(d) for d in oracle_dir.iterdir() if d.is_dir() and d.name != ROUND)) \
    if oracle_dir.exists() else set()
n = 30 if ROUND == "pilot" else cfg["data"]["n_oracle_labels"]
rows = sample_for_labelling(items, gens, n, cfg["languages"]["targets"], cfg["languages"]["reference"],
                            splits, cfg["project"]["seed"], exclude=earlier)
paths = export_round(rows, oracle_dir / ROUND, ANNOTATORS, cfg["project"]["seed"])
from collections import Counter
print(len(rows), "rows, excluding", len(earlier), "from earlier rounds:",
      dict(Counter((r["lang"], r["objective"]) for r in rows)))
for lang, ps in paths.items():
    print(lang, [str(p) for p in ps])

## B · Agreement

After both annotators of a language have put their sheets in `filled/`. Continue to the main round only if kappa ≥ `oracle.min_kappa` (0.6) on **both** fields; otherwise discuss the disagreements, tighten `docs/oracle_protocol.md` (log the change at the bottom) and run a new pilot round (`ROUND = "pilot2"`).

In [ ]:
from mpac.data.oracle import agreement, load_round, resolve

ROUND = "pilot"
for lang, (labels, adj) in load_round(subdir(cfg, "oracle") / ROUND).items():
    if not labels:
        print(lang, ": no filled sheets yet"); continue
    a = agreement(labels)
    _, unresolved = resolve(labels, adj)
    ok = all(v["kappa"] >= cfg["oracle"]["min_kappa"] for v in a.values())
    print(f"{lang}: {len(labels)} labels from {len({l.annotator for l in labels})} annotators | "
          f"unresolved {len(unresolved)} | {'OK' if ok else 'BELOW min_kappa'}")
    for field, v in a.items():
        print(f"   {field:24s} kappa {v['kappa']:.2f}  raw agreement {v['raw_agreement']:.2f}  n {v['n_items']:.0f}")

## C · Table 6: evaluator systems on the same outputs

- `SOURCE = "proxy"` works today (pipeline check, **not reportable**); `"human"` reads the resolved `main` round.
- Thresholds start at each language's LaBSE floor + 0.15 (options: 0.3) and are tuned per (language, signal) on **tune** labels only; the table shows which were tuned.
- The adaptive run sets the budget; the single evaluator and the fixed team get the same number of evaluations. Scores use **test** labels and count labelled items a system never looked at as missed.
- `wrong_language_responses`: free-text responses fastText says aren't in the prompt's language. Agents flag those as collapse.

In [ ]:
import pandas as pd
from mpac.analysis.evaluators import table6

SOURCE = "proxy"        # "human" once the main round is resolved
CHECKPOINT = "base"

out = table6(cfg, m["id"], CHECKPOINT, source=SOURCE)
print(out["oracle"], "\nbudget:", out["budget"], "evaluations | wrong-language responses:",
      out["wrong_language_responses"], "\nsaved ->", out["path"], "\n")
cols = ["precision", "recall", "f1", "n_labelled", "n_labelled_evaluated", "n_evaluations", "cells", "tokens"]
print(pd.DataFrame(out["systems"]).T[cols].round(3).to_string(), "\n")
print(pd.DataFrame(out["thresholds"]).T[["threshold", "source", "n_labels", "n_collapsed"]].to_string(), "\n")
a = out["adaptive"]
print("adaptive population by cycle:", a["population_by_cycle"], "| ops:", a["lifecycle_ops"])
print("final agents:", a["final_alive"])